# cascade: evidence, confidence and edges

All the work happens in `compute/evidence.py`; each cell below calls one function. It runs in SageMaker AI with the notebook's IAM role, reads `s3://$BUCKET/raw/manual_docs/` and `s3://$BUCKET/processed/`, and calls Amazon Bedrock. Simulation only, not advice.

Part A builds the evidence table. Part B builds the confidence labels and the graph edges.

## Setup
Set the environment variables and import the module. `EVIDENCE_MODEL_ID` defaults to `global.anthropic.claude-sonnet-5-5`.

In [ ]:
%env BUCKET=cascade-s3-bucket
%env AWS_REGION=us-east-1
%env EVIDENCE_MODEL_ID=global.anthropic.claude-sonnet-5-5

In [ ]:
import logging, sys
sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO, format="%(message)s")
from compute import evidence

## Part A, steps 1-5: read the documents and extract evidence
Reads `manifest_docs.csv` and each listed PDF or HTML file, splits the text into ~1,500-character passages (200 overlap), keeps up to 25 keyword-matching passages per document, and asks Bedrock for a JSON verdict on each. A row is kept only if its excerpt is a verbatim substring of the passage. At most 3 rows are kept per (ticker, channel). Token usage totals are logged at the end.

Documents of doc_type `research` (ticker `macro`, or empty = macro) are read for two extra topics, `currency` and `stock_bond`, with their own keywords. The manifest may have optional columns `topic`, `publisher` and `date`; without `topic`, a research document's topic is inferred from its text. Research rows get `evidence_id` `<ticker_or_macro>-<topic>-<n>` (for example `macro-stock_bond-1`). Only documents with no cache under `processed/_evidence_cache/` (new or changed manifest rows) are sent to the model. Every evidence row has a `publisher`, a `date` (`not stated` if unknown), a `source_url` and a `trust` tier (Central bank, Research, Fund issuer, Company filing or Press).

In [ ]:
usage = evidence.Usage()
ev_df, passages = evidence.build_evidence(usage=usage)
ev_df

## Part A, steps 6-7: write the evidence table and Knowledge Base documents
Writes `processed/evidence/part-0.parquet`, plus `evidence/<evidence_id>.txt` and `<evidence_id>.txt.metadata.json` for each row. Old objects under `evidence/` are removed first.

In [ ]:
keys = evidence.write_evidence(ev_df, passages)
print(len(ev_df), "evidence rows;", len(keys), "objects written")

## Part B, steps 8-12: confidence labels and edges
Reads `processed/sensitivities`, `chain_reconciliation`, `links` and `evidence`. Assigns each ticker its main channel, applies the contract label rules, and writes `processed/confidence` and `processed/edges`. Besides the measured edges and the `oil -> <ticker>` cited edges, research evidence gives cited edges `risk_sentiment -> usdcad` (topic currency) and `inflation -> stock_bond` (topic stock_bond), each linked to its `evidence_id`.

In [ ]:
out = evidence.run_part_b()
out["confidence"]

In [ ]:
out["edges"]

## Checks
Every excerpt must appear in its source text, RY.TO must be grey, and the number of `.metadata.json` files under `evidence/` must equal the evidence row count. Every evidence row must have a publisher, a date, a source_url and a trust tier.

In [ ]:
checks = evidence.check_outputs()
checks

In [ ]:
assert (checks["excerpts_ok"] and checks["ry_grey"] and checks["metadata_count_ok"]
        and checks["provenance_ok"]), checks
usage.log_totals()